# 12 — Troubleshooting, performance and session management

**Goal:** diagnose the problems met most often, with a check for each. Run this notebook first when
something fails.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nisar_s1.config import load_kit
from nisar_s1 import product as P

kit = load_kit()                                  # training/notebooks/kit_config.yaml
OUT = Path(kit["data_dir"]) / "outputs" / "12"   # outputs stay outside the repository
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})

In [2]:
import sys, importlib, platform, os
rows = [{"item": "python", "value": sys.version.split()[0]}, {"item": "platform", "value": platform.platform()}]
for m in ["nisar_s1", "h5py", "numpy", "rasterio", "geopandas", "pyproj", "earthaccess", "folium"]:
    try:
        mod = importlib.import_module(m); rows.append({"item": m, "value": getattr(mod, "__version__", "ok")})
    except Exception as e:
        rows.append({"item": m, "value": f"MISSING ({type(e).__name__})"})
pd.DataFrame(rows)

/home/dell/anaconda3/envs/nisar_s1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,item,value
0,python,3.11.18
1,platform,Linux-7.0.0-43-generic-x86_64-with-glibc2.39
2,nisar_s1,0.2.0
3,h5py,3.16.0
4,numpy,2.4.6
5,rasterio,1.4.4
6,geopandas,1.2.0
7,pyproj,3.7.2
8,earthaccess,0.17.0
9,folium,1.0.0


## Typical problems

| Symptom | Cause | Check / fix |
|---|---|---|
| `FileNotFoundError: no session file` | notebooks run out of order | run 01, 02, 06 first |
| `KeyError: session file is missing [...]` | an earlier step was skipped | the message names the missing keys |
| `MemoryError: window needs ... GB` | AOI too large for the guard | shrink the AOI (06) or process by tiles (05) |
| `401 / 403` when streaming | no or wrong Earthdata credentials | `~/.netrc` with `urs.earthdata.nasa.gov`; accept the NISAR EULA on the Earthdata profile |
| `FileNotFoundError` on an ASF URL | product moved collection (e.g. `..._PROVISIONAL_V1`) | search the granule again (`asf_search.granule_search`) and update the URL |
| `PROJ: proj_create_from_database ...` printed by GDAL | two PROJ databases in the environment | harmless if coordinates are right (notebook 10 checks it); set `PROJ_DATA` to `pyproj.datadir.get_data_dir()` |
| very slow reads | strided reads or windows not aligned on chunks | read windows, see notebook 05 |
| kernel dies | RAM exhausted | close other notebooks; lower `max_gb`; on Linux run long jobs with `systemd-run --user -p MemoryMax=...` |

In [3]:
netrc = Path.home() / ".netrc"
ok_netrc = netrc.exists() and "urs.earthdata.nasa.gov" in netrc.read_text()   # content never printed
print("Earthdata entry in ~/.netrc:", ok_netrc)
import pyproj
print("pyproj data dir:", pyproj.datadir.get_data_dir())
print("PROJ_DATA env  :", os.environ.get("PROJ_DATA", "(not set)"))

Earthdata entry in ~/.netrc: True
pyproj data dir: /home/dell/anaconda3/envs/nisar_s1/share/proj
PROJ_DATA env  : (not set)


## Session file: inspect, and reset if needed

In [4]:
sf = Path(kit["session_file"])
s = P.load_session(sf) if sf.exists() else {}
for k, v in s.items():
    print(f"{k:20s} {str(v)[:90]}")
RESET = False            # set True to start again from notebook 01
if RESET:
    sf.unlink(); print("session reset")

granule              NISAR_L2_PR_GCOV_034_144_A_026_2005_DHDH_M_20261103T041937_20261103T042013_P05036_N_F_J_00
gcov_local           /media/dell/Transcend1/nisar_data/kit/gcov/NISAR_L2_PR_GCOV_034_144_A_026_2005_DHDH_M_2026
gcov_url             https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_V1/NISAR_L2_PR_GCOV_034_144_
product_type         GCOV
epsg_hint            32632
frequency            frequencyA
epsg                 32632
polarimetric_mode    dual-pol linear (HH/HV)
terms                ['HHHH', 'HVHV']
aoi_name             vosges_c11
aoi_bbox_wgs84       [6.716, 47.885, 6.797, 47.935]
aoi_bounds           [329250.0627272764, 5305867.839017732, 335463.8838031237, 5311600.571279069]
aoi_window           {'rows': [11295, 11583], 'cols': [4726, 5038], 'transform': [20.0, 0.0, 329240.0, 0.0, -20
qc_valid_fraction    1.0
gslc_subsets         ['/media/dell/Transcend1/nisar_data/kit/gslc/NISAR_L2_PR_GSLC_025_144_A_026_2005_DHDH_M_20
c2_looks             [8, 4]


## Memory available now

In [5]:
try:
    import psutil
    vm = psutil.virtual_memory(); print(f"RAM available: {vm.available / 2**30:.1f} of {vm.total / 2**30:.1f} GB")
except ImportError:
    print(Path("/proc/meminfo").read_text().splitlines()[2] if Path("/proc/meminfo").exists() else "unknown")

RAM available: 0.7 of 7.7 GB


In [6]:
checks = {
    "package importable": rows[2]["value"] != "MISSING",
    "session readable": isinstance(s, dict),
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")
assert all(checks.values()), "some checks failed: read the messages above"
print("Module 12 STATUS: PASS")

  [x] package importable
  [x] session readable
Module 12 STATUS: PASS
